# Preprocessing Pipeline Sanity Checks

This notebook verifies the preprocessing pipeline end-to-end:
1. Generate synthetic data
2. Normalise signals
3. Compute STFT spectrograms
4. Resize to model input size
5. Convert to tensors
6. Run PCA dimensionality reduction
7. Verify shapes and value ranges at each step

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.data_loading import load_or_generate_synthetic, save_processed_dataset
from src.preprocessing import (
    batch_preprocess, compute_fft, extract_tabular_features,
    augment_signal, standardize_features,
)
from src.spectrograms import stft_spectrogram, resize_spectrogram, spectrogram_to_tensor
from src.dimensionality_reduction import fit_transform_pca
from src.utils import set_seed

set_seed(42)
print('All imports OK')

## Step 1: Generate Synthetic Data

In [ ]:
data = load_or_generate_synthetic(n_classes=4, n_samples_per_class=50, signal_length=512, seed=42)

# Flatten to arrays
X_list, y_list = [], []
for i, (name, arr) in enumerate(data.items()):
    X_list.append(arr)
    y_list.append(np.full(len(arr), i))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'X range: [{X.min():.3f}, {X.max():.3f}]')
assert not np.any(np.isnan(X)), 'NaN in data!'
print('✓ No NaN values')

## Step 2: FFT Sanity Check

In [ ]:
sig = X[0]
mag = compute_fft(sig, n_fft=512)

print(f'Signal length: {len(sig)}')
print(f'FFT magnitude shape: {mag.shape}')
print(f'FFT magnitude range: [{mag.min():.4f}, {mag.max():.4f}]')
assert np.all(mag >= 0), 'Magnitude should be non-negative'
print('✓ FFT magnitudes are non-negative')

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(sig)
axes[0].set_title('Raw IQ Signal')
axes[0].set_xlabel('Sample')

freqs = np.fft.rfftfreq(512)
axes[1].plot(freqs, mag)
axes[1].set_title('Magnitude Spectrum')
axes[1].set_xlabel('Normalised Frequency')
plt.tight_layout()
plt.show()

## Step 3: STFT Spectrogram

In [ ]:
spec = stft_spectrogram(sig, n_fft=128, hop_length=32, normalize=True)

print(f'Spectrogram shape: {spec.shape}')
print(f'Spectrogram range: [{spec.min():.4f}, {spec.max():.4f}]')
print(f'Dtype: {spec.dtype}')
assert spec.dtype == np.float32, 'Spectrogram should be float32'
assert spec.min() >= -0.01 and spec.max() <= 1.01, 'Normalised range check failed'
print('✓ Spectrogram shape, dtype and value range OK')

plt.figure(figsize=(8, 4))
plt.imshow(spec, aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='Normalised power (dB)')
plt.title('STFT Spectrogram')
plt.xlabel('Time frame')
plt.ylabel('Frequency bin')
plt.tight_layout()
plt.show()

## Step 4: Resize Spectrogram

In [ ]:
target_size = (64, 64)
spec_resized = resize_spectrogram(spec, target_size=target_size)

print(f'Original shape: {spec.shape} → Resized: {spec_resized.shape}')
assert spec_resized.shape == target_size, f'Expected {target_size}, got {spec_resized.shape}'
print(f'✓ Resized to {target_size}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(spec, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title(f'Original {spec.shape}')
axes[1].imshow(spec_resized, aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title(f'Resized {spec_resized.shape}')
plt.tight_layout()
plt.show()

## Step 5: Convert to PyTorch Tensor

In [ ]:
import torch

tensor = spectrogram_to_tensor(spec_resized, normalize=True)

print(f'Tensor shape: {tensor.shape}')  # Should be (3, 64, 64)
print(f'Tensor dtype: {tensor.dtype}')
print(f'Tensor range: [{tensor.min().item():.4f}, {tensor.max().item():.4f}]')
assert tensor.shape == (3, 64, 64), f'Expected (3,64,64), got {tensor.shape}'
assert 0.0 <= tensor.min().item() and tensor.max().item() <= 1.0 + 1e-5
print('✓ Tensor shape, dtype and value range OK')

## Step 6: Batch Preprocessing

In [ ]:
specs_batch = batch_preprocess(X[:10], n_fft=128, hop_length=32, augment=False)
print(f'Batch spectrogram shape: {specs_batch.shape}')
assert specs_batch.ndim == 3, 'Expected 3D array (N, freq, time)'
assert specs_batch.shape[0] == 10
print('✓ Batch preprocessing OK')

# With augmentation
specs_aug = batch_preprocess(X[:10], n_fft=128, hop_length=32, augment=True)
assert specs_aug.shape == specs_batch.shape
print('✓ Batch preprocessing with augmentation OK')

## Step 7: Tabular Feature Extraction

In [ ]:
tab_feats = np.stack([extract_tabular_features(s) for s in X])
print(f'Tabular features shape: {tab_feats.shape}')
assert tab_feats.shape == (len(X), 12), f'Expected (N,12), got {tab_feats.shape}'
assert np.all(np.isfinite(tab_feats)), 'Non-finite values in tabular features'
print('✓ Tabular features: shape and finiteness OK')

# Standardise
tab_std, mean, std = standardize_features(tab_feats)
print(f'\nAfter standardisation:')
print(f'  Mean (per feature): {tab_std.mean(axis=0).round(4)}')
print(f'  Std (per feature): {tab_std.std(axis=0).round(4)}')
np.testing.assert_allclose(tab_std.mean(axis=0), np.zeros(12), atol=1e-4)
np.testing.assert_allclose(tab_std.std(axis=0), np.ones(12), atol=1e-3)
print('✓ Standardisation: zero mean, unit std OK')

## Step 8: PCA Dimensionality Reduction

In [ ]:
n_components = 8
X_pca, pca = fit_transform_pca(tab_feats, n_components=n_components)

print(f'Input shape: {tab_feats.shape}')
print(f'PCA output shape: {X_pca.shape}')
print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(3)}')
print(f'Total explained variance: {pca.explained_variance_ratio_.sum():.3f}')
assert X_pca.shape == (len(X), n_components)
print(f'✓ PCA reduction from {tab_feats.shape[1]} → {n_components} OK')

# Visualise PCA variance
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, n_components+1), pca.explained_variance_ratio_, label='Per component')
ax.step(range(1, n_components+1), np.cumsum(pca.explained_variance_ratio_),
        where='mid', color='red', label='Cumulative')
ax.axhline(0.95, ls='--', color='gray', alpha=0.7, label='95% threshold')
ax.set_xlabel('PCA Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA Explained Variance')
ax.legend()
plt.tight_layout()
plt.show()

## Step 9: PCA 2D Scatter (visualise class separation)

In [ ]:
from sklearn.decomposition import PCA

pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(tab_feats)

colors_cls = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
class_names_list = list(data.keys())

fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, color) in enumerate(zip(class_names_list, colors_cls)):
    mask = y == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], alpha=0.7, label=name,
               color=color, s=30, edgecolors='white', linewidths=0.5)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA 2D Projection of Tabular Features')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## ✅ Sanity Check Summary

| Step | Check | Status |
|------|-------|--------|
| 1. Data Loading | Shape, NaN | ✓ |
| 2. FFT | Magnitude non-negative | ✓ |
| 3. STFT Spectrogram | Shape, range [0,1], float32 | ✓ |
| 4. Resize | Target size achieved | ✓ |
| 5. Tensor Conversion | Shape (3,H,W), range [0,1] | ✓ |
| 6. Batch Preprocess | 3D output, augmentation | ✓ |
| 7. Tabular Features | Shape (N,12), finite | ✓ |
| 8. Standardisation | Zero mean, unit std | ✓ |
| 9. PCA | Shape (N, n_components) | ✓ |